# Phase 1: Exploratory Data Analysis

## Section 1 — Load and Inspect

In [ ]:
import os
import pandas as pd
import numpy as np

REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.isdir(os.path.join(REPO_DIR, 'data')):
    REPO_DIR = os.getcwd()
TRAIN = {l1: os.path.join(REPO_DIR, f'data/train/{l1}/kvl_shared_task_{l1}_train.csv') for l1 in ['es','de','cn']}
DEV = {l1: os.path.join(REPO_DIR, f'data/dev/{l1}/kvl_shared_task_{l1}_dev.csv') for l1 in ['es','de','cn']}

dfs = {}
for l1 in ['es','de','cn']:
    tr = pd.read_csv(TRAIN[l1])
    dv = pd.read_csv(DEV[l1])
    dfs[f'train_{l1}'] = tr
    dfs[f'dev_{l1}'] = dv
    print(f'--- {l1} train ---')
    print('shape:', tr.shape)
    print('dtypes:', tr.dtypes)
    print(tr.head(3))
    print('nulls:', tr.isnull().sum())
    print('duplicates:', tr.duplicated(subset=['item_id']).sum())
    print()

In [ ]:
tr_es, tr_de, tr_cn = dfs['train_es'], dfs['train_de'], dfs['train_cn']
common = set(tr_es['item_id']) & set(tr_de['item_id']) & set(tr_cn['item_id'])
print('Same item_id across L1s = same English word. Common item_ids:', len(common))

## Section 2 — GLMM_score Distribution per L1
**CRITICAL: Lower GLMM_score = MORE difficult.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (l1, key) in zip(axes, [('es','train_es'), ('de','train_de'), ('cn','train_cn')]):
    df = dfs[key]
    df['GLMM_score'].hist(ax=ax, density=True, alpha=0.6, label='Hist')
    df['GLMM_score'].plot.kde(ax=ax, label='KDE')
    ax.set_title(f'L1={l1}')
    ax.legend()
    ax.set_xlabel('GLMM_score (lower = harder)')
plt.tight_layout()
plt.savefig(os.path.join(REPO_DIR, 'results/figures/glmm_distribution.pdf'), bbox_inches='tight')
plt.show()
for l1 in ['es','de','cn']:
    print(dfs[f'train_{l1}']['GLMM_score'].describe())

In [ ]:
for l1 in ['es','de','cn']:
    df = dfs[f'train_{l1}'].sort_values('GLMM_score', ascending=True)
    print(f'10 hardest words ({l1}):')
    print(df[['item_id','en_target_word','GLMM_score']].head(10))
    print()

## Section 3 — Cross-L1 Score Correlation

In [ ]:
m = tr_es[['item_id','GLMM_score']].rename(columns={'GLMM_score':'GLMM_es'})
m = m.merge(tr_de[['item_id','GLMM_score']].rename(columns={'GLMM_score':'GLMM_de'}), on='item_id')
m = m.merge(tr_cn[['item_id','GLMM_score']].rename(columns={'GLMM_score':'GLMM_cn'}), on='item_id')
print('Pearson es-de:', m['GLMM_es'].corr(m['GLMM_de']))
print('Pearson es-cn:', m['GLMM_es'].corr(m['GLMM_cn']))
print('Pearson de-cn:', m['GLMM_de'].corr(m['GLMM_cn']))
sns.heatmap(m[['GLMM_es','GLMM_de','GLMM_cn']].corr(), annot=True, cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Cross-L1 GLMM_score correlation')
plt.tight_layout()
plt.savefig(os.path.join(REPO_DIR, 'results/figures/cross_l1_correlation.pdf'), bbox_inches='tight')
plt.show()

## Section 4 — Cognate Analysis (es, de only)

In [ ]:
import Levenshtein

def norm_edit(s1, s2):
    if not s1 or not s2: return 1.0
    return Levenshtein.distance(str(s1), str(s2)) / max(len(str(s1)), len(str(s2)), 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, l1 in zip(axes, ['es', 'de']):
    df = dfs[f'train_{l1}'].copy()
    df['edit_dist'] = df.apply(lambda r: norm_edit(r['en_target_word'], r['L1_source_word']), axis=1)
    r = df['edit_dist'].corr(df['GLMM_score'])
    print(f'{l1} Pearson(edit_distance, GLMM_score): {r:.4f} (NEGATIVE r is correct: higher distance = harder = lower GLMM_score)')
    ax.scatter(df['edit_dist'], df['GLMM_score'], alpha=0.3)
    ax.set_xlabel('Normalized edit distance')
    ax.set_ylabel('GLMM_score')
    ax.set_title(f'L1={l1}')
plt.suptitle('Cognate: edit_distance vs GLMM_score (es, de)')
plt.tight_layout()
plt.savefig(os.path.join(REPO_DIR, 'results/figures/cognate_scatter.pdf'), bbox_inches='tight')
plt.show()

## Section 5 — Clue Informativeness

In [ ]:
def reveal_ratio(row):
    w = str(row.get('en_target_word',''))
    c = str(row.get('en_target_clue',''))
    if not w: return 0
    revealed = sum(1 for x in c if x not in ('_',' ') and x.strip())
    return revealed / len(w)

for l1 in ['es','de','cn']:
    df = dfs[f'train_{l1}'].copy()
    df['reveal_ratio'] = df.apply(reveal_ratio, axis=1)
    r = df['reveal_ratio'].corr(df['GLMM_score'])
    print(f'{l1} Pearson(reveal_ratio, GLMM_score): {r:.4f}')
plt.figure(figsize=(6,4))
df = dfs['train_es'].copy()
df['reveal_ratio'] = df.apply(reveal_ratio, axis=1)
plt.scatter(df['reveal_ratio'], df['GLMM_score'], alpha=0.3)
plt.xlabel('reveal_ratio')
plt.ylabel('GLMM_score')
plt.title('Clue informativeness vs GLMM_score')
plt.savefig(os.path.join(REPO_DIR, 'results/figures/clue_reveal.pdf'), bbox_inches='tight')
plt.show()

## Section 6 — POS and Word Properties

In [ ]:
pos_means = []
for l1 in ['es','de','cn']:
    df = dfs[f'train_{l1}']
    gm = df.groupby('en_target_pos')['GLMM_score'].mean()
    for pos, val in gm.items():
        pos_means.append({'L1': l1, 'POS': pos, 'mean_GLMM': val})
pos_df = pd.DataFrame(pos_means)
pivot = pos_df.pivot(index='POS', columns='L1', values='mean_GLMM')
pivot.plot(kind='bar', figsize=(10,4))
plt.ylabel('Mean GLMM_score')
plt.title('Mean GLMM_score by POS and L1')
plt.legend(title='L1')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(REPO_DIR, 'results/figures/pos_difficulty.pdf'), bbox_inches='tight')
plt.show()

## Section 7 — Context Complexity (within L1 only)

In [ ]:
for l1 in ['es','de','cn']:
    df = dfs[f'train_{l1}'].copy()
    df['ctx_len'] = df['L1_context'].fillna('').str.split().str.len()
    df['ctx_ttr'] = df['L1_context'].fillna('').apply(lambda t: len(set(t.split()))/max(len(t.split()),1))
    df['ctx_avg_wlen'] = df['L1_context'].fillna('').apply(lambda t: np.mean([len(w) for w in t.split()]) if t.split() else 0)
    for col in ['ctx_len','ctx_ttr','ctx_avg_wlen']:
        r = df[col].corr(df['GLMM_score'])
        print(f'{l1} {col} vs GLMM_score: r={r:.4f}')
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, col in zip(axes, ['ctx_len','ctx_ttr','ctx_avg_wlen']):
        ax.scatter(df[col], df['GLMM_score'], alpha=0.3)
        ax.set_xlabel(col)
        ax.set_ylabel('GLMM_score')
    plt.suptitle(f'Context complexity ({l1})')
    plt.tight_layout()
    plt.savefig(os.path.join(REPO_DIR, f'results/figures/context_complexity_{l1}.pdf'), bbox_inches='tight')
    plt.show()